# Cohort tumor and PerturbView decoding QC

Read an exported `cohort_cell_analysis.h5ad` without loading its intensity matrix into memory. This notebook summarizes persisted per-cell decoding and ZNCC alignment fields, visualizes tumor assignments, optional tissue-artifact annotations, minimum ZNCC, and called guides in slide-local micron coordinates, and provides a selected-slide zoom. It does not rerun decoding or modify the H5AD.

Tumor polygons are not embedded in the cohort H5AD, so tumor maps show the spatial cloud of cells assigned to each tumor rather than the original GeoJSON boundary.

In [ ]:
from pathlib import Path
import re

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300})

## Inputs

Backed mode leaves the large cell-by-channel intensity matrix (`X` and layers) on disk. Observation annotations and spatial coordinates are loaded because they are the data being inspected.

In [ ]:
COHORT_H5AD = Path("/path/to/post_analysis/cohort_cell_analysis.h5ad")
NO_CALL_LABEL = "None"
UNKNOWN_LABEL = "UNK"
UNASSIGNED_TUMOR = "unassigned"
ALIGNMENT_REFERENCE_CHANNEL = None  # Exact alias, or None for metadata/autodetection
MAX_POINTS_PER_SLIDE = 300_000
NIMBUS_QC_CHUNK_ROWS = 250_000
POINT_SIZE = 0.25
RANDOM_SEED = 0

cohort = ad.read_h5ad(COHORT_H5AD, backed="r")
obs = cohort.obs
spatial = np.asarray(cohort.obsm["spatial"], dtype=np.float64)

required = {"slide_id", "tumor_id", "decode_eligible", "decode_incomplete_input", "decode_guide_call"}
missing = sorted(required.difference(obs.columns))
if missing:
    raise KeyError(f"Cohort H5AD is missing required observation fields: {missing}")
if spatial.shape != (cohort.n_obs, 2):
    raise ValueError(f"Expected obsm['spatial'] shape {(cohort.n_obs, 2)}, got {spatial.shape}")
if "nimbus" not in cohort.layers:
    raise KeyError("Cohort is missing required layers['nimbus'].")
if "nimbus_available" not in cohort.var.columns:
    raise KeyError("Cohort var is missing required nimbus_available mask.")
nimbus_available = cohort.var["nimbus_available"].fillna(False).to_numpy(dtype=bool)
if not nimbus_available.any():
    raise ValueError("Cohort nimbus_available mask selects no channels.")
nimbus_columns = cohort.var_names[nimbus_available].astype(str).tolist()
cohort_metadata = cohort.uns.get("post_analysis_cohort", {})
metadata_nimbus_columns = [str(value) for value in cohort_metadata.get("nimbus_columns", [])]
if set(metadata_nimbus_columns) != set(nimbus_columns):
    missing_from_mask = sorted(set(metadata_nimbus_columns) - set(nimbus_columns))
    missing_from_metadata = sorted(set(nimbus_columns) - set(metadata_nimbus_columns))
    raise ValueError(
        "Nimbus aliases in var['nimbus_available'] disagree with cohort metadata: "
        f"missing_from_mask={missing_from_mask}, missing_from_metadata={missing_from_metadata}"
    )
if metadata_nimbus_columns != nimbus_columns:
    print("Nimbus metadata and padded layer contain the same aliases in different orders; using intensity-axis order.")

round_names = []
for column in obs.columns:
    match = re.fullmatch(r"decode_(.+)_pass", str(column))
    if match:
        round_names.append(match.group(1))
if not round_names:
    raise KeyError("No per-round decode_*_pass fields were found.")

slide_values = obs["slide_id"].astype(str).to_numpy()
tumor_values = obs["tumor_id"].astype(str).to_numpy()
guide_values = obs["decode_guide_call"].astype(str).to_numpy()
eligible = obs["decode_eligible"].fillna(False).to_numpy(dtype=bool)
incomplete = obs["decode_incomplete_input"].fillna(True).to_numpy(dtype=bool)
if "tissue_artifact" in obs.columns:
    artifact_values = pd.to_numeric(obs["tissue_artifact"], errors="coerce").to_numpy(dtype=float)
    artifact_annotated = np.isfinite(artifact_values)
    invalid_artifact = artifact_annotated & ~np.isin(artifact_values, [0.0, 1.0])
    if invalid_artifact.any():
        raise ValueError(f"tissue_artifact contains {int(invalid_artifact.sum()):,} values other than 0, 1, or NaN")
    tissue_artifact = artifact_values == 1.0
else:
    artifact_values = np.full(cohort.n_obs, np.nan, dtype=float)
    artifact_annotated = np.zeros(cohort.n_obs, dtype=bool)
    tissue_artifact = np.zeros(cohort.n_obs, dtype=bool)
tumor_assigned = tumor_values != UNASSIGNED_TUMOR
any_call = eligible & (guide_values != NO_CALL_LABEL)
mapped_call = any_call & (guide_values != UNKNOWN_LABEL)
unknown_call = any_call & (guide_values == UNKNOWN_LABEL)
all_rounds_pass = eligible & np.logical_and.reduce([
    obs[f"decode_{name}_pass"].fillna(False).to_numpy(dtype=bool)
    for name in round_names
])

print(cohort)
print(f"Slides: {pd.unique(slide_values).tolist()}")
print(f"Detected decoding rounds: {round_names}")
print(f"Nimbus channels: {nimbus_columns}")
print(f"Tissue-artifact annotation available for {int(artifact_annotated.sum()):,}/{cohort.n_obs:,} cells")

## Cohort channel and Nimbus-layer validation

Cohort generation already requires identical ordered intensity and Nimbus channels across slides. This check verifies the persisted padded layer without materializing it all at once: expected Nimbus channels must be finite, while channels outside the Nimbus subset must be exactly zero.

In [ ]:
nimbus_layer = cohort.layers["nimbus"]
if nimbus_layer.shape != cohort.shape:
    raise ValueError(f"Nimbus layer shape {nimbus_layer.shape} does not match cohort shape {cohort.shape}.")

expected_nonfinite = 0
padding_nonzero = 0
padding_nonfinite = 0
for start in range(0, cohort.n_obs, NIMBUS_QC_CHUNK_ROWS):
    stop = min(cohort.n_obs, start + NIMBUS_QC_CHUNK_ROWS)
    values = np.asarray(nimbus_layer[start:stop, :], dtype=np.float32)
    expected = values[:, nimbus_available]
    padding = values[:, ~nimbus_available]
    expected_nonfinite += int((~np.isfinite(expected)).sum())
    padding_nonfinite += int((~np.isfinite(padding)).sum())
    padding_nonzero += int((np.isfinite(padding) & (padding != 0)).sum())

nimbus_qc = pd.Series({
    "n_cells": cohort.n_obs,
    "n_intensity_channels": cohort.n_vars,
    "n_nimbus_channels": int(nimbus_available.sum()),
    "n_zero_padded_channels": int((~nimbus_available).sum()),
    "expected_nimbus_nonfinite_values": expected_nonfinite,
    "padding_nonzero_values": padding_nonzero,
    "padding_nonfinite_values": padding_nonfinite,
})
display(nimbus_qc)
if expected_nonfinite or padding_nonzero or padding_nonfinite:
    raise ValueError("Nimbus layer failed finite-value or zero-padding validation.")

## ZNCC alignment QC

Summarize pre-warp ZNCC correlation across non-reference rounds. `minimum_zncc` is the lowest finite comparison-round correlation for each cell, so low values identify cells whose worst-aligned round is poor. The explicit reference column is excluded; cells with no valid comparison-round value remain `NaN`. No threshold or filtering is applied.

In [ ]:
if "alignment_zncc" not in cohort.obsm:
    raise KeyError("Cohort is missing required obsm['alignment_zncc'].")
alignment_block = cohort.obsm["alignment_zncc"]
if isinstance(alignment_block, pd.DataFrame):
    if not alignment_block.index.equals(cohort.obs_names):
        raise ValueError("alignment_zncc rows are not aligned to cohort.obs_names.")
    alignment_columns = alignment_block.columns.astype(str).tolist()
    alignment_values = alignment_block.to_numpy(dtype=np.float32, copy=False)
else:
    alignment_values = np.asarray(alignment_block, dtype=np.float32)
    alignment_columns = [str(value) for value in cohort_metadata.get("alignment_columns", [])]
if alignment_values.shape != (cohort.n_obs, len(alignment_columns)):
    raise ValueError(
        f"Alignment matrix shape {alignment_values.shape} does not match "
        f"({cohort.n_obs}, {len(alignment_columns)})."
    )
if len(set(alignment_columns)) != len(alignment_columns):
    raise ValueError("Alignment aliases must be unique.")

reference_channel = ALIGNMENT_REFERENCE_CHANNEL
if reference_channel is None:
    stored_reference = cohort_metadata.get("alignment_reference_channel")
    if stored_reference is not None and str(stored_reference) not in {"", "None"}:
        reference_channel = str(stored_reference)
if reference_channel is None:
    reference_candidates = []
    for column_index, column in enumerate(alignment_columns):
        values = alignment_values[:, column_index]
        if np.isfinite(values).all() and np.allclose(values, 1.0, rtol=0, atol=1e-6):
            reference_candidates.append(column)
    if len(reference_candidates) != 1:
        raise ValueError(
            "Could not identify exactly one all-ones ZNCC reference column; set "
            f"ALIGNMENT_REFERENCE_CHANNEL explicitly. Candidates: {reference_candidates}"
        )
    reference_channel = reference_candidates[0]
if reference_channel not in alignment_columns:
    raise KeyError(f"Alignment reference {reference_channel!r} is absent: {alignment_columns}")

comparison_columns = [column for column in alignment_columns if column != reference_channel]
if not comparison_columns:
    raise ValueError("At least one non-reference alignment channel is required.")
comparison_positions = [alignment_columns.index(column) for column in comparison_columns]
alignment_valid_round_count = np.zeros(cohort.n_obs, dtype=np.int16)
minimum_zncc = np.full(cohort.n_obs, np.inf, dtype=np.float32)
for column, column_index in zip(comparison_columns, comparison_positions):
    values = alignment_values[:, column_index]
    finite = np.isfinite(values)
    if ((values[finite] < -1.000001) | (values[finite] > 1.000001)).any():
        raise ValueError(f"Finite ZNCC correlations for {column!r} must lie in [-1, 1].")
    alignment_valid_round_count += finite
    minimum_zncc[finite] = np.minimum(minimum_zncc[finite], values[finite])
has_valid_alignment = alignment_valid_round_count > 0
minimum_zncc[~has_valid_alignment] = np.nan

alignment_cell_qc = pd.DataFrame({
    "slide_id": slide_values,
    "minimum_zncc": minimum_zncc,
    "n_valid_comparison_rounds": alignment_valid_round_count,
    "complete_comparison_rounds": alignment_valid_round_count == len(comparison_columns),
    "no_valid_comparison_round": alignment_valid_round_count == 0,
})
alignment_min_summary = alignment_cell_qc.groupby("slide_id", sort=False).agg(
    n_cells=("minimum_zncc", "size"),
    n_finite=("minimum_zncc", "count"),
    median_minimum_zncc=("minimum_zncc", "median"),
    n_complete_comparison_rounds=("complete_comparison_rounds", "sum"),
    n_no_valid_comparison_round=("no_valid_comparison_round", "sum"),
)
minimum_groups = alignment_cell_qc.groupby("slide_id", sort=False)["minimum_zncc"]
alignment_min_summary["q05_minimum_zncc"] = minimum_groups.quantile(0.05)
alignment_min_summary["q10_minimum_zncc"] = minimum_groups.quantile(0.10)
alignment_min_summary["q25_minimum_zncc"] = minimum_groups.quantile(0.25)
alignment_min_summary["fraction_nan"] = 1 - alignment_min_summary["n_finite"] / alignment_min_summary["n_cells"]
alignment_min_summary["fraction_complete_comparison_rounds"] = alignment_min_summary["n_complete_comparison_rounds"] / alignment_min_summary["n_cells"]
display(pd.Series({"reference_channel": reference_channel, "comparison_rounds": len(comparison_columns)}))
display(alignment_min_summary)

alignment_round_rows = []
for column, column_index in zip(comparison_columns, comparison_positions):
    frame = pd.DataFrame({"slide_id": slide_values, "zncc": alignment_values[:, column_index]})
    grouped = frame.groupby("slide_id", sort=False)["zncc"].agg(
        n_cells="size", n_finite="count", median="median"
    ).reset_index()
    grouped["q10"] = frame.groupby("slide_id", sort=False)["zncc"].quantile(0.10).to_numpy()
    grouped["fraction_nan"] = 1 - grouped["n_finite"] / grouped["n_cells"]
    grouped.insert(1, "alignment_channel", column)
    alignment_round_rows.append(grouped)
alignment_round_summary = pd.concat(alignment_round_rows, ignore_index=True)
display(alignment_round_summary)

slide_order_for_hist = pd.unique(slide_values).tolist()
fig, axes = plt.subplots(
    len(slide_order_for_hist), 1, figsize=(9, 3.5 * len(slide_order_for_hist)), squeeze=False
)
bins = np.linspace(-1, 1, 101)
for row, slide_id in enumerate(slide_order_for_hist):
    values = minimum_zncc[(slide_values == slide_id) & np.isfinite(minimum_zncc)]
    axes[row, 0].hist(values, bins=bins, color="steelblue", log=True)
    axes[row, 0].set(xlim=(-1, 1), xlabel="Minimum ZNCC across comparison rounds", ylabel="Cells", title=str(slide_id))
fig.tight_layout()
plt.show()

## Compact decoding metrics

The primary denominator is `eligible_nuclear_cells`: tumor-assigned cells with complete finite nuclear decoding measurements. `any_call` includes both mapped guides and `UNK`; `mapped_guide` excludes `None` and `UNK`.

In [ ]:
flags = pd.DataFrame({
    "slide_id": slide_values,
    "total_cells": True,
    "tumor_assigned": tumor_assigned,
    "artifact_annotated_cells": artifact_annotated,
    "tissue_artifact_cells": tissue_artifact,
    "complete_nuclear_input": ~incomplete,
    "eligible_nuclear_cells": eligible,
    "all_rounds_pass": all_rounds_pass,
    "any_call": any_call,
    "mapped_guide": mapped_call,
    "unknown_tuple": unknown_call,
})
counts = flags.groupby("slide_id", sort=False).sum().astype(np.int64)
counts.loc["COHORT"] = flags.drop(columns="slide_id").sum().astype(np.int64)
counts["fraction_eligible_among_tumor"] = counts["eligible_nuclear_cells"] / counts["tumor_assigned"]
counts["fraction_tissue_artifact_among_annotated"] = counts["tissue_artifact_cells"] / counts["artifact_annotated_cells"]
counts["fraction_any_call_among_eligible"] = counts["any_call"] / counts["eligible_nuclear_cells"]
counts["fraction_mapped_guide_among_eligible"] = counts["mapped_guide"] / counts["eligible_nuclear_cells"]
counts["fraction_unknown_among_eligible"] = counts["unknown_tuple"] / counts["eligible_nuclear_cells"]
counts["fraction_all_rounds_pass_among_eligible"] = counts["all_rounds_pass"] / counts["eligible_nuclear_cells"]
display(counts)

In [ ]:
round_rows = []
eligible_index = np.flatnonzero(eligible)
for round_name in round_names:
    frame = pd.DataFrame({
        "slide_id": slide_values[eligible_index],
        "pass_raw_threshold": obs[f"decode_{round_name}_pass_top"].iloc[eligible_index].to_numpy(dtype=bool),
        "pass_ratio": obs[f"decode_{round_name}_pass_ratio"].iloc[eligible_index].to_numpy(dtype=bool),
        "pass_both": obs[f"decode_{round_name}_pass"].iloc[eligible_index].to_numpy(dtype=bool),
        "ratio": obs[f"decode_{round_name}_ratio"].iloc[eligible_index].to_numpy(dtype=float),
        "top_fold": obs[f"decode_{round_name}_top_fold"].iloc[eligible_index].to_numpy(dtype=float),
    })
    grouped = frame.groupby("slide_id", sort=False).agg(
        n_eligible=("pass_both", "size"),
        pass_raw_threshold=("pass_raw_threshold", "mean"),
        pass_ratio=("pass_ratio", "mean"),
        pass_both=("pass_both", "mean"),
        median_ratio=("ratio", "median"),
        median_top_fold=("top_fold", "median"),
    ).reset_index()
    grouped.insert(1, "round", round_name)
    round_rows.append(grouped)
round_qc = pd.concat(round_rows, ignore_index=True)
display(round_qc)

## Guide composition by tumor

`None` and `UNK` are excluded. Fractions are normalized within each slide/tumor combination over mapped guide calls.

In [ ]:
called_index = np.flatnonzero(mapped_call)
guide_counts = (
    pd.DataFrame({
        "slide_id": slide_values[called_index],
        "tumor_id": tumor_values[called_index],
        "guide": guide_values[called_index],
    })
    .groupby(["slide_id", "tumor_id", "guide"], observed=True, sort=False)
    .size().rename("n_cells").reset_index()
)
guide_counts["fraction_within_tumor_calls"] = (
    guide_counts["n_cells"]
    / guide_counts.groupby(["slide_id", "tumor_id"], observed=True)["n_cells"].transform("sum")
)
display(guide_counts.sort_values(["slide_id", "tumor_id", "n_cells"], ascending=[True, True, False]))

In [ ]:
matrix = guide_counts.pivot_table(
    index=["slide_id", "tumor_id"], columns="guide",
    values="fraction_within_tumor_calls", fill_value=0.0,
)
fig, ax = plt.subplots(figsize=(max(10, 0.25 * matrix.shape[1]), max(3, 0.35 * matrix.shape[0])))
image = ax.imshow(matrix.to_numpy(), aspect="auto", cmap="viridis", vmin=0)
ax.set_xticks(np.arange(matrix.shape[1]), matrix.columns, rotation=90, fontsize=7)
ax.set_yticks(np.arange(matrix.shape[0]), [f"{slide} | {tumor}" for slide, tumor in matrix.index], fontsize=7)
fig.colorbar(image, ax=ax, label="Fraction of mapped calls in tumor")
ax.set_title("Guide composition by tumor")
fig.tight_layout()
plt.show()

## Whole-slide tumor, tissue-artifact, minimum-ZNCC, and guide maps

Visualization is reproducibly subsampled after selecting each population. Summaries above always use every cell. Artifact maps show reviewed non-artifact cells in light gray and artifact cells in red; unannotated slides are labeled explicitly. Minimum-ZNCC maps use a fixed `[-1, 1]` scale, with invalid cells in light gray. Colors are fixed cohort-wide so a guide has the same color on every slide.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
slide_order = pd.unique(slide_values).tolist()
tumor_categories = sorted(pd.unique(tumor_values[tumor_assigned]).tolist())
guide_categories = sorted(pd.unique(guide_values[mapped_call]).tolist())
tumor_colors = {name: plt.cm.turbo(i / max(1, len(tumor_categories) - 1)) for i, name in enumerate(tumor_categories)}
guide_colors = {name: plt.cm.turbo(i / max(1, len(guide_categories) - 1)) for i, name in enumerate(guide_categories)}

def sampled_indices(mask, maximum=MAX_POINTS_PER_SLIDE):
    indices = np.flatnonzero(mask)
    if len(indices) > maximum:
        indices = rng.choice(indices, size=maximum, replace=False)
    return indices

def scatter_categories(ax, indices, values, colors, *, size=POINT_SIZE):
    for value in pd.unique(values[indices]):
        selected = indices[values[indices] == value]
        ax.scatter(spatial[selected, 0], spatial[selected, 1], s=size, color=colors[value], linewidths=0, rasterized=True)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")

def scatter_artifact_status(ax, mask, *, size=POINT_SIZE):
    reviewed = mask & artifact_annotated
    if not reviewed.any():
        ax.text(0.5, 0.5, "Not artifact-annotated", ha="center", va="center", transform=ax.transAxes)
    else:
        background_index = sampled_indices(reviewed & ~tissue_artifact)
        artifact_index = sampled_indices(reviewed & tissue_artifact)
        ax.scatter(spatial[background_index, 0], spatial[background_index, 1], s=size, color="0.82", linewidths=0, rasterized=True)
        ax.scatter(spatial[artifact_index, 0], spatial[artifact_index, 1], s=max(size, 0.5), color="red", linewidths=0, rasterized=True)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")

def scatter_minimum_zncc(ax, mask, *, size=POINT_SIZE):
    missing_index = sampled_indices(mask & ~np.isfinite(minimum_zncc))
    finite_index = sampled_indices(mask & np.isfinite(minimum_zncc))
    ax.scatter(spatial[missing_index, 0], spatial[missing_index, 1], s=size, color="0.82", linewidths=0, rasterized=True)
    artist = ax.scatter(
        spatial[finite_index, 0], spatial[finite_index, 1], c=minimum_zncc[finite_index],
        s=size, cmap="viridis", vmin=-1, vmax=1, linewidths=0, rasterized=True,
    )
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
    return artist

fig, axes = plt.subplots(len(slide_order), 4, figsize=(26, 6 * len(slide_order)), squeeze=False)
for row, slide_id in enumerate(slide_order):
    on_slide = slide_values == slide_id
    tumor_index = sampled_indices(on_slide & tumor_assigned)
    guide_index = sampled_indices(on_slide & mapped_call)
    scatter_categories(axes[row, 0], tumor_index, tumor_values, tumor_colors)
    scatter_artifact_status(axes[row, 1], on_slide)
    zncc_artist = scatter_minimum_zncc(axes[row, 2], on_slide)
    scatter_categories(axes[row, 3], guide_index, guide_values, guide_colors)
    axes[row, 0].set_title(f"{slide_id}: tumor-assigned cells")
    axes[row, 1].set_title(f"{slide_id}: tissue artifacts")
    axes[row, 2].set_title(f"{slide_id}: minimum ZNCC")
    axes[row, 3].set_title(f"{slide_id}: mapped guide calls (None/UNK hidden)")
    fig.colorbar(zncc_artist, ax=axes[row, 2], label="Minimum ZNCC", shrink=0.75)
fig.tight_layout()
plt.show()

## Per-round ZNCC maps for every slide

Display comparison rounds in their persisted acquisition order. Every panel for a slide uses the same reproducible cell sample and the same `[-1, 1]` color scale, so spatial degradation can be followed from round to round. Light-gray cells are invalid for that round. Panel titles report the finite-cell median and valid fraction.

In [ ]:
def plot_round_alignment_maps(slide_id, *, columns_per_row=4, size=POINT_SIZE):
    on_slide = slide_values == str(slide_id)
    if not on_slide.any():
        raise KeyError(f"Slide {slide_id!r} is absent from the cohort.")
    plot_index = sampled_indices(on_slide)
    n_columns = min(int(columns_per_row), len(comparison_columns))
    n_rows = (len(comparison_columns) + n_columns - 1) // n_columns
    fig, axes = plt.subplots(
        n_rows, n_columns, figsize=(5.2 * n_columns, 5.0 * n_rows), squeeze=False
    )
    flat_axes = axes.ravel()
    artist = None
    for panel, (column, column_index) in enumerate(zip(comparison_columns, comparison_positions)):
        ax = flat_axes[panel]
        values = alignment_values[plot_index, column_index]
        finite = np.isfinite(values)
        missing_index = plot_index[~finite]
        finite_index = plot_index[finite]
        ax.scatter(
            spatial[missing_index, 0], spatial[missing_index, 1],
            s=size, color="0.82", linewidths=0, rasterized=True,
        )
        artist = ax.scatter(
            spatial[finite_index, 0], spatial[finite_index, 1], c=values[finite],
            s=size, cmap="viridis", vmin=-1, vmax=1, linewidths=0, rasterized=True,
        )
        median = float(np.median(values[finite])) if finite.any() else np.nan
        ax.set_title(f"{column}\nmedian={median:.3f}; valid={finite.mean():.3%}")
        ax.invert_yaxis()
        ax.set_aspect("equal")
        ax.set_xlabel("x (µm)")
        ax.set_ylabel("y (µm)")
    for ax in flat_axes[len(comparison_columns):]:
        ax.set_visible(False)
    fig.suptitle(f"{slide_id}: ZNCC correlation by comparison round", fontsize=14)
    fig.colorbar(
        artist, ax=flat_axes[:len(comparison_columns)].tolist(),
        label="ZNCC correlation", shrink=0.75, pad=0.02,
    )
    fig.subplots_adjust(top=0.92, right=0.92, hspace=0.28, wspace=0.22)
    return fig, axes

for slide_id in slide_order:
    plot_round_alignment_maps(slide_id)
    plt.show()

## Selected-slide zoom

Specify the slide and global micron-coordinate bounds here. Zoom controls intentionally live next to the plot.

In [ ]:
SELECTED_SLIDE = slide_order[0]
ZOOM_UM = None  # (xmin, xmax, ymin, ymax), or None for the complete slide

on_slide = slide_values == SELECTED_SLIDE
if ZOOM_UM is not None:
    xmin, xmax, ymin, ymax = map(float, ZOOM_UM)
    in_zoom = (spatial[:, 0] >= xmin) & (spatial[:, 0] <= xmax) & (spatial[:, 1] >= ymin) & (spatial[:, 1] <= ymax)
else:
    in_zoom = np.ones(cohort.n_obs, dtype=bool)

fig, axes = plt.subplots(1, 4, figsize=(26, 7))
tumor_index = sampled_indices(on_slide & in_zoom & tumor_assigned)
guide_index = sampled_indices(on_slide & in_zoom & mapped_call)
scatter_categories(axes[0], tumor_index, tumor_values, tumor_colors, size=0.5)
scatter_artifact_status(axes[1], on_slide & in_zoom, size=0.5)
zncc_artist = scatter_minimum_zncc(axes[2], on_slide & in_zoom, size=0.5)
scatter_categories(axes[3], guide_index, guide_values, guide_colors, size=0.5)
axes[0].set_title(f"{SELECTED_SLIDE}: tumors")
axes[1].set_title(f"{SELECTED_SLIDE}: tissue artifacts")
axes[2].set_title(f"{SELECTED_SLIDE}: minimum ZNCC")
axes[3].set_title(f"{SELECTED_SLIDE}: mapped guides")
fig.colorbar(zncc_artist, ax=axes[2], label="Minimum ZNCC", shrink=0.75)
fig.tight_layout()
plt.show()

## What was persisted

The cohort `obs` contains all cell-level decoding outputs: eligibility, missing/incomplete nuclear input, each round's winner index, ratio, raw winner intensity, threshold, threshold fold, quality and pass flags, plus the decoded tuple and guide call. These support empirical cohort, slide, tumor and round QC. When tissue-artifact annotation was supplied, `obs['tissue_artifact']` contains `0` outside and `1` inside artifact regions. In cohorts mixing annotated and unannotated slides, unannotated cells are `NaN`, not `0`; if no slide was annotated, the column is absent and this notebook reports every slide as unannotated.

Nimbus is stored in `layers['nimbus']` on the shared intensity channel axis. `var['nimbus_available']` selects the actual Nimbus channels; all other layer columns are structural zero padding. Cohort construction requires identical ordered intensity and Nimbus channel lists across slides and finite values for every expected Nimbus measurement. Subset to `nimbus_available` before PCA, scaling, neighbors, or clustering.

Alignment correlation remains in `obsm['alignment_zncc']`. This notebook derives `minimum_zncc` in memory as the minimum finite correlation across non-reference rounds; it does not write the derived metric back to the cohort H5AD or apply an exclusion threshold.

Per-slide fitted threshold/scaling dictionaries are not consolidated into `cohort.uns`. They remain in each slide's `decode_settings.json` and per-slide H5AD `uns['post_analysis']`; compact `decode_funnel.csv`, `guide_counts.csv`, and `guide_counts_by_tumor.csv` files are also exported beside each slide H5AD.